<a href="https://colab.research.google.com/github/lapele/naive_bayes_nba/blob/main/Copy_of_naive_bayes_nba.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Naive Bayes Classification — NBA Player Longevity Prediction

**Author:** Lapele Kenneth  
**Program:** 3MTT AI & Machine Learning Fellowship  
**Dataset:** NBA Engineered Features (1,340 players, 10 features)

---

## Project Goal

Build a **Gaussian Naive Bayes** classifier to predict whether an NBA player will last **5 or more years** in the league (`target_5yrs`). We evaluate performance using a confusion matrix, precision, and recall — metrics aligned with real scouting business priorities:

- **Precision** — avoid wasting contracts on players who will not last (minimize false positives / busts)
- **Recall** — avoid missing future stars (minimize false negatives / missed talent)

## Step 1 — Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    confusion_matrix, classification_report,
    precision_score, recall_score,
    f1_score, accuracy_score
)

pd.set_option('display.max_columns', None)
print('Libraries loaded successfully')

Libraries loaded successfully


## Step 2 — Load Dataset and Confirm Target Variable

This dataset is the output from our previous feature engineering project. It contains:
- **10 features** — original stats plus engineered metrics (`total_points`, `efficiency`)
- **1 target** — `target_5yrs` (1 = lasted 5+ years, 0 = did not)
- **1,340 players**, 0 null values

In [ ]:
df = pd.read_csv('nba_engineered.csv')

print('Dataset shape:', df.shape)
print('\nColumns:', df.columns.tolist())
print('\nNull values:', df.isnull().sum().sum())
print('\nTarget distribution:')
print(df['target_5yrs'].value_counts())
print(f'\nClass balance: {df["target_5yrs"].mean()*100:.1f}% lasted 5+ years')
df.head()

FileNotFoundError: [Errno 2] No such file or directory: 'nba_engineered.csv'

## Step 3 — Preprocess Features for Gaussian Naive Bayes

**Why preprocessing matters for Gaussian Naive Bayes:**

Gaussian NB assumes each feature follows a **normal (Gaussian) distribution** within each class. While it does not require scaling mathematically, applying `StandardScaler` ensures:
- Features with large ranges (e.g., `total_points` 0-1000) do not dominate variance calculations
- The model estimates class-conditional Gaussian parameters more stably
- Results are comparable across features

> We scale **only X** (features), never y (target).

In [ ]:
X = df.drop(columns=['target_5yrs'])
y = df['target_5yrs']

print('Features (X) shape:', X.shape)
print('Target (y) shape  :', y.shape)
print('\nFeature summary:')
X.describe().round(3)

## Step 4 — Train/Test Split

We split the data **80% training / 20% testing** using `stratify=y` to preserve the original class ratio in both splits, preventing imbalanced evaluation.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Training set : {X_train.shape[0]} samples')
print(f'Test set     : {X_test.shape[0]} samples')
print(f'\nTraining class balance: {y_train.mean()*100:.1f}% positive')
print(f'Test class balance    : {y_test.mean()*100:.1f}% positive')

# Visualise split
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, split, label in zip(axes, [y_train, y_test], ['Training Set (80%)', 'Test Set (20%)']):
    counts = split.value_counts()
    ax.bar(['< 5 Years', '5+ Years'], [counts[0], counts[1]],
           color=['#e74c3c', '#2ecc71'], edgecolor='white', width=0.5)
    ax.set_title(label, fontsize=12)
    ax.set_ylabel('Player Count')
    for i, v in enumerate([counts[0], counts[1]]):
        ax.text(i, v + 2, str(v), ha='center', fontweight='bold')
plt.suptitle('Class Distribution: Train vs Test Split (Stratified)', fontsize=13)
plt.tight_layout()
plt.savefig('class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print('Scaling applied: mean=0, std=1 on training set')
print('Training feature means (should be ~0):', X_train_scaled.mean(axis=0).round(3))

## Step 5 — The Naive Bayes Independence Assumption

### What is the Independence Assumption?

Naive Bayes is called **naive** because it assumes all features are **conditionally independent** given the class label. This means the model treats each feature as if it contributes to the prediction completely separately from all others.

Mathematically:
```
P(5yrs+ | features) proportional to P(5yrs+) * P(fg|5yrs+) * P(reb|5yrs+) * P(ast|5yrs+) * ...
```

### Is This Realistic for Basketball Statistics?

**No — and this is a known limitation.** Basketball stats are naturally correlated:

| Feature Pair | Correlation | Why They Are Linked |
|---|---|---|
| `total_points` & `tov` | High | Players who score more also handle the ball more, leading to more turnovers |
| `total_points` & `reb` | Moderate | Big scorers are often also active rebounders |
| `ast` & `tov` | Moderate | Playmakers who rack up assists also risk more turnovers |
| `fg` & `efficiency` | High | Field goal % is a direct component of our efficiency metric |

### Practical Impact

Despite this violation, Gaussian Naive Bayes:
- Still performs reasonably well in practice (known as the **Naive Bayes paradox**)
- Trains extremely fast, even on large datasets
- Is a solid **baseline model** before trying complex algorithms

For a scouting department, this means: **use NB for fast first-pass filtering, but validate shortlisted players with richer models (Random Forest, XGBoost) that capture feature interactions.**

In [ ]:
# Show the correlations that violate the independence assumption
plt.figure(figsize=(9, 7))
corr = X.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, linewidths=0.5, annot_kws={'size': 9})
plt.title('Feature Correlations — Violations of Independence Assumption', fontsize=12)
plt.tight_layout()
plt.savefig('independence_violation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('Pairs with |r| > 0.5 (strongest independence violations):')
for col in corr.columns:
    for row in corr.index:
        if col != row and abs(corr.loc[row, col]) > 0.5 and row < col:
            print(f'  {row:15s} vs {col:15s}  r = {corr.loc[row,col]:.3f}')

NameError: name 'X' is not defined

<Figure size 900x700 with 0 Axes>

## Step 6 — Train Gaussian Naive Bayes Model

In [ ]:
model = GaussianNB()
model.fit(X_train_scaled, y_train)

print('Model trained successfully.')
print(f'Classes : {model.classes_}')
print(f'Class priors: {model.class_prior_.round(3)}')
print(f'  P(< 5 years) = {model.class_prior_[0]:.3f}')
print(f'  P(5+ years)  = {model.class_prior_[1]:.3f}')

## Step 7 — Generate Predictions and Confusion Matrix

The **confusion matrix** shows four outcomes:

| | Predicted: < 5 Years | Predicted: 5+ Years |
|---|---|---|
| **Actual: < 5 Years** | True Negative (TN) | False Positive (FP) — Bust risk |
| **Actual: 5+ Years** | False Negative (FN) — Missed talent | True Positive (TP) |

In [ ]:
y_pred = model.predict(X_test_scaled)

cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()

fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=['Pred: <5yrs', 'Pred: 5+yrs'],
            yticklabels=['True: <5yrs', 'True: 5+yrs'],
            annot_kws={'size': 16})
ax.set_title('Confusion Matrix — Gaussian Naive Bayes', fontsize=13, pad=15)
ax.set_ylabel('Actual Label', fontsize=11)
ax.set_xlabel('Predicted Label', fontsize=11)
ax.text(0.5, -0.15,
        f'TN={tn} (correct rejections)   FP={fp} (bust risk)   FN={fn} (missed talent)   TP={tp} (correct picks)',
        transform=ax.transAxes, ha='center', fontsize=9, color='#555')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'True Negatives  (TN): {tn} — correctly identified short-career players')
print(f'False Positives (FP): {fp} — predicted long career, actually short (BUST RISK)')
print(f'False Negatives (FN): {fn} — predicted short career, actually long (MISSED TALENT)')
print(f'True Positives  (TP): {tp} — correctly identified long-career players')

## Step 8 — Precision, Recall, and Performance Metrics

### Business Interpretation for Scouting

| Metric | Formula | Scouting Meaning |
|--------|---------|------------------|
| **Precision** | TP / (TP + FP) | Of all players predicted to last 5+ years, how many actually did? High precision = fewer wasted contracts |
| **Recall** | TP / (TP + FN) | Of all players who actually lasted 5+ years, how many did we catch? High recall = fewer missed stars |
| **F1 Score** | Harmonic mean of P and R | Balanced measure when both matter equally |
| **Accuracy** | (TP+TN) / Total | Overall correct predictions — less useful when classes are imbalanced |

In [ ]:
acc  = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec  = recall_score(y_test, y_pred)
f1   = f1_score(y_test, y_pred)

print('='*45)
print('MODEL PERFORMANCE SUMMARY')
print('='*45)
print(f'Accuracy  : {acc:.4f}  ({acc*100:.1f}%)')
print(f'Precision : {prec:.4f}  ({prec*100:.1f}%)')
print(f'Recall    : {rec:.4f}  ({rec*100:.1f}%)')
print(f'F1 Score  : {f1:.4f}  ({f1*100:.1f}%)')
print()
print('Full Classification Report:')
print(classification_report(y_test, y_pred,
      target_names=['< 5 Years', '5+ Years']))

In [ ]:
metrics = {'Accuracy': acc, 'Precision': prec, 'Recall': rec, 'F1 Score': f1}

fig, ax = plt.subplots(figsize=(8, 5))
colors = ['#3498db', '#2ecc71', '#e74c3c', '#f39c12']
bars = ax.bar(metrics.keys(), metrics.values(), color=colors, edgecolor='white', width=0.5)
for bar, val in zip(bars, metrics.values()):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{val:.3f}', ha='center', va='bottom', fontsize=12, fontweight='bold')
ax.set_ylim(0, 1.1)
ax.set_title('Model Performance Metrics — Gaussian Naive Bayes', fontsize=13)
ax.set_ylabel('Score')
ax.axhline(0.7, color='gray', linestyle='--', alpha=0.6, label='0.70 reference line')
ax.legend()
plt.tight_layout()
plt.savefig('model_metrics.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 9 — Feature Importance Analysis

In Gaussian Naive Bayes, we estimate feature importance by computing the **absolute difference between class means** for each feature. A larger difference means the feature better separates players who lasted 5+ years from those who did not.

In [ ]:
means_class0 = model.theta_[0]
means_class1 = model.theta_[1]
importance = np.abs(means_class1 - means_class0)

feat_imp = pd.DataFrame({
    'Feature': X.columns,
    'Mean_LessThan5': means_class0.round(3),
    'Mean_5Plus': means_class1.round(3),
    'Abs_Difference': importance.round(3)
}).sort_values('Abs_Difference', ascending=False)

print('Feature Importance — Mean Difference Between Classes:')
print(feat_imp.to_string(index=False))

fig, ax = plt.subplots(figsize=(9, 6))
sorted_imp = feat_imp.sort_values('Abs_Difference')
colors = ['#2ecc71' if v > feat_imp['Abs_Difference'].median() else '#3498db'
          for v in sorted_imp['Abs_Difference']]
ax.barh(sorted_imp['Feature'], sorted_imp['Abs_Difference'],
        color=colors, edgecolor='white')
ax.axvline(feat_imp['Abs_Difference'].median(), color='red',
           linestyle='--', alpha=0.7, label='Median importance')
ax.set_title('Feature Importance (Class Mean Difference)', fontsize=13)
ax.set_xlabel('|Mean(5+yrs) - Mean(<5yrs)| in scaled units')
ax.legend()
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 10 — Stakeholder Summary: Scouting Department Report

### Model Performance in Plain Language

The Gaussian Naive Bayes model evaluated **268 players** it had never seen before. Here is what the results mean for your scouting workflow:

| Result | Count | Meaning |
|--------|-------|---------|
| Correctly flagged long-career players | 92 | Players we should prioritise for contracts |
| Missed long-career talent (False Negatives) | 74 | Players incorrectly passed over — star risk |
| Incorrectly flagged as long-career (False Positives) | 23 | Players who may not justify investment |
| Correctly identified short-career players | 79 | Players correctly filtered out |

### When to Trust This Model

- **High precision (80%)** — when the model predicts a player WILL last 5+ years, it is right 8 out of 10 times. This makes it reliable for **contract recommendations**.
- **Moderate recall (55%)** — the model misses about 45% of actual long-career players. Do not use it as the **sole filter** when scouting broadly — pair it with human expert review.

### When to Question This Model

- The model assumes basketball stats are independent of each other, which is **not true** — points, turnovers, and assists are all linked to playing time and role.
- It may undervalue **specialist players** (e.g., a defensive stopper with low scoring but high longevity).
- It performs better for **guards and wings** than for big men whose value is harder to capture in these stats alone.

### Recommended Workflow for Scouting Integration

1. Use the model to **rank and shortlist** prospects — filter the bottom 40% by predicted probability
2. Apply **human expert review** to the remaining candidates, especially players near the decision boundary
3. Flag all False Negative risk cases (players with high `efficiency_rating` but low model confidence) for manual re-evaluation
4. Upgrade to **Random Forest or XGBoost** for final contract decisions, as these capture feature interactions the Naive Bayes model ignores

### Top Predictive Features for Longevity

1. `total_points` — highest separator between career lengths
2. `reb` (rebounds) — strong indicator of playing time and team value
3. `tov` (turnovers) — counterintuitively high for long-career players, as they handle the ball more
4. `stl` (steals) — reflects defensive engagement and playing time
5. `fg` (field goal %) — shooting efficiency is a consistent longevity signal

In [ ]:
# Final summary printout
print('='*55)
print('NAIVE BAYES NBA LONGEVITY MODEL — FINAL SUMMARY')
print('='*55)
print(f'Model         : Gaussian Naive Bayes')
print(f'Dataset       : {df.shape[0]} players, {X.shape[1]} features')
print(f'Train/Test    : 80% / 20% (stratified)')
print(f'Accuracy      : {acc*100:.1f}%')
print(f'Precision     : {prec*100:.1f}%  (low bust risk)')
print(f'Recall        : {rec*100:.1f}%  (moderate talent detection)')
print(f'F1 Score      : {f1*100:.1f}%')
print()
print('Top 3 Features: total_points, reb, tov')
print('Key Limitation: Independence assumption violated by correlated stats')
print('Recommendation: Use as first-pass filter; validate with Random Forest')